# SETU Compliance Policy Assistant

Multi-agent RAG chatbot over 48 SETU institutional policies.  
Uses **Groq** (LLM) + **Jina AI** (embeddings) + **ChromaDB** (vector store) + **Gradio** (UI).

### Before you start — add your API keys as Colab Secrets
1. Click the **🔑 key icon** in the left sidebar
2. Add these secrets (all free-tier):
   | Name | Get it from |
   |------|-------------|
   | `GROQ_API_KEY` | https://console.groq.com/keys |
   | `JINA_API_KEY` | https://jina.ai |
   | `CEREBRAS_API_KEY` | https://cloud.cerebras.ai *(optional fallback)* |
3. Enable **"Notebook access"** toggle for each secret
4. Run cells top-to-bottom

## Step 1 — Clone the repository

In [ ]:
import os

REPO_DIR = "/content/setu-compliance-rag"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/krbuti/setu-compliance-rag.git {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

## Step 2 — Install dependencies

Takes ~2 minutes. GPU runtime is not required but speeds up the cross-encoder reranker.

In [ ]:
%%capture install_log
# Core RAG stack
!pip install -q \
    langchain==1.2.7 \
    langchain-core==1.4.1 \
    langchain-text-splitters \
    langchain-community==0.4.2 \
    langchain-openai==1.1.7 \
    langchain-anthropic \
    langchain-groq \
    langchain-huggingface \
    langgraph==1.0.10 \
    langgraph-prebuilt==1.0.8 \
    chromadb==1.5.8

# Retrieval utilities
!pip install -q \
    sentence-transformers \
    rank_bm25 \
    einops \
    python-dotenv

# Gradio UI
!pip install -q gradio

# Ollama stub (imported by langchain-ollama — not used in Colab)
!pip install -q langchain-ollama==1.0.1

print("All packages installed")

In [ ]:
# Quick sanity check
import importlib
required = [
    "langchain", "langgraph", "chromadb",
    "sentence_transformers", "rank_bm25", "gradio"
]
missing = [m for m in required if not importlib.util.find_spec(m)]
if missing:
    print(f"MISSING: {missing} — re-run the install cell above")
else:
    print("All core packages available")

## Step 3 — Configure API keys

Reads from Colab Secrets (preferred) or falls back to manual entry.

In [ ]:
import os

def _get_secret(name: str, fallback: str = "") -> str:
    """Try Colab Secrets first, then env, then return empty string."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name, fallback)

# ── Fill these in if you didn't set Colab Secrets ───────────────────────────
GROQ_API_KEY     = _get_secret("GROQ_API_KEY")     or ""  # paste key here
JINA_API_KEY     = _get_secret("JINA_API_KEY")     or ""  # paste key here
CEREBRAS_API_KEY = _get_secret("CEREBRAS_API_KEY") or ""  # optional
# ────────────────────────────────────────────────────────────────────────────

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY is missing!\n"
        "Add it as a Colab Secret (key icon in the sidebar) "
        "or paste it directly into the cell above."
    )

if not JINA_API_KEY:
    raise ValueError(
        "JINA_API_KEY is missing!\n"
        "Get a free key at https://jina.ai and add it as a Colab Secret."
    )

# Inject into the environment so all modules pick them up
os.environ["LLM_PROVIDER"]        = "groq"
os.environ["LLM_MODEL"]           = "llama-3.1-8b"
os.environ["EMBEDDINGS_PROVIDER"] = "jina"
os.environ["EMBEDDINGS_MODEL"]    = "jina-embeddings-v2-base-en"
os.environ["GROQ_API_KEY"]        = GROQ_API_KEY
os.environ["JINA_API_KEY"]        = JINA_API_KEY
if CEREBRAS_API_KEY:
    os.environ["CEREBRAS_API_KEY"] = CEREBRAS_API_KEY

print(f"LLM  : Groq / llama-3.1-8b-instant")
print(f"Embed: Jina AI / jina-embeddings-v2-base-en")
print(f"GROQ key  : {'set (' + GROQ_API_KEY[:8] + '...)' if GROQ_API_KEY else 'MISSING'}")
print(f"Jina key  : {'set (' + JINA_API_KEY[:8] + '...)' if JINA_API_KEY else 'MISSING'}")

## Step 4 — Build the vector database

Reads the 48 pre-extracted policy JSONs from `dataset/` and indexes 3,394 chunks into ChromaDB.
Uses Jina cloud embeddings (GPU-backed, ~1-2 minutes).

In [ ]:
from pathlib import Path

CHROMA_DIR = Path("/content/setu-compliance-rag/chroma_db")

if CHROMA_DIR.exists() and any(CHROMA_DIR.iterdir()):
    print(f"ChromaDB already exists at {CHROMA_DIR} — skipping ingest.")
    print("Delete the folder and re-run this cell to force a full re-ingest.")
else:
    print("Building ChromaDB from dataset JSONs...")
    import sys
    sys.path.insert(0, "/content/setu-compliance-rag")
    from ingest import ingest
    n = ingest()
    print(f"Done — {n} chunks indexed.")

## Step 5 — Load the agent

Initialises the LangGraph pipeline and BM25 index (takes ~30 seconds on first run).

In [ ]:
import sys
sys.path.insert(0, "/content/setu-compliance-rag")

from main_agent import handle_query
from information_agent import load_vectorstore

# Warm up: load vectorstore + BM25 index + cross-encoder
vs = load_vectorstore()
print("\nAgent ready — running a test query...")
test_answer = handle_query("What is the maternity leave entitlement for SETU staff?")
print(f"\nTest answer:\n{test_answer}")

## Step 6 — Launch Gradio chat interface

Generates a **public shareable link** valid for 72 hours. Share it with anyone who should be able to test the chatbot.

In [ ]:
import gradio as gr

DISCLAIMER = (
    "**Academic prototype — not an official SETU service.**  \n"
    "Answers are generated from 48 SETU policy documents using AI and may be "
    "incomplete or inaccurate. Always verify with the official policy document "
    "or contact your HR / relevant SETU office before making decisions."
)

def chat(message: str, history: list) -> str:
    if not message.strip():
        return "Please enter a question about SETU policies."
    answer = handle_query(message)
    return answer + "\n\n---\n*Disclaimer: Academic AI prototype. Verify with official SETU documents or HR before acting on this information.*"

demo = gr.ChatInterface(
    fn=chat,
    title="SETU Compliance Policy Assistant",
    description=(
        "Ask questions about SETU's 48 institutional policies — "
        "leave entitlements, recruitment, EDI, research conduct, "
        "data protection, Gen AI, and more.\n\n" + DISCLAIMER
    ),
    examples=[
        "How many weeks of maternity leave is a female staff member entitled to?",
        "What is SETU's data protection policy?",
        "Is Garda vetting required for all staff roles?",
        "What constitutes research misconduct at SETU?",
        "What does the Gen AI policy say about staff using AI tools?",
        "What are the leave entitlements for adoptive parents at SETU?",
        "How does SETU handle complaints about workplace harassment?",
        "What is SETU's policy on intellectual property for staff research?",
    ],
    cache_examples=False,
)

# share=True creates a public Gradio link (72h)
demo.launch(share=True, debug=False)

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `GROQ_API_KEY missing` | Add it in Colab Secrets (🔑 sidebar) and re-run Step 3 |
| `ChromaDB not found` | Re-run Step 4 |
| `ModuleNotFoundError` | Re-run Step 2 (install), then restart runtime |
| Gradio link expired | Re-run Step 6 (new 72h link issued) |
| Slow first query | Normal — cross-encoder model downloads on first use (~100 MB) |
| Runtime disconnected | Re-run Steps 4→5→6 (ChromaDB is rebuilt in Colab /content which resets on disconnect) |

### Switching to a better LLM
In Step 3, change:
```python
os.environ["LLM_MODEL"] = "llama-3.3-70b"  # Groq's largest free model
```
Then re-run Steps 3 and 5.